In [ ]:
from transformers import pipeline 
fill = pipeline("fill-mask", model = "distilbert-base-uncased")
print(fill("The patient was admitted to the [MASK]"))


In [ ]:
from datasets import load_dataset
dataset = load_dataset("imdb", split = "train")
print(dataset)
print(dataset[0])

In [ ]:
from transformers import AutoTokenizer 
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
inputs = tokenizer(list(dataset["text"]), return_tensors = "pt", padding = True , truncation = True)

In [ ]:
print(inputs)

{'input_ids': tensor([[  101,  1045, 12524,  ...,     0,     0,     0],
        [  101,  1000,  1045,  ...,     0,     0,     0],
        [  101,  2065,  2069,  ...,     0,     0,     0],
        ...,
        [  101,  2023,  2143,  ...,     0,     0,     0],
        [  101,  1005,  1996,  ...,  2108,  8487,   102],
        [  101,  1996,  2466,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]])}


In [ ]:
print(inputs["input_ids"].shape)

torch.Size([25000, 512])


In [ ]:
encodings = tokenizer(list(dataset["text"]), return_tensors = "pt", padding = True , truncation = True)

In [ ]:
import torch
labels = torch.tensor(dataset["label"])

In [ ]:
from sklearn.model_selection import train_test_split  


train_idx, test_idx  = train_test_split(range(len(labels)) , test_size = 0.2, random_state = 44)

train_encodings = {key: val[train_idx] for key, val in encodings.items()}
test_encodings = {key: val[test_idx] for key, val in encodings.items()}
train_labels = labels[train_idx]
test_labels = labels[test_idx]

In [ ]:
from torch.utils.data import Dataset 

class IMDBDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings 
        self.labels = labels 
    def __len__(self):
        return len(self.labels)
    
        
    def __getitem__(self,idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx]
        }
        
        
        
        

In [ ]:
train_obj = IMDBDataset(train_encodings, train_labels)
test_obj = IMDBDataset(test_encodings, test_labels)


In [ ]:
from torch.utils.data import DataLoader 

train_loader = DataLoader(train_obj, batch_size = 8)
test_loader = DataLoader(test_obj, batch_size = 8)

In [ ]:
from transformers import AutoModelForSequenceClassification 
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels = 2)


In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr = 5e-5)

In [ ]:
device = torch.device("cuda")
model = model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:


model.train()

for epoch in range(3):
    for batch in train_loader:
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        loss.backward()
        optimizer.step()
    
    print(f"Epoch {epoch + 1}, loss: {loss.item()}")

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0)) 

True
Tesla T4


In [ ]:
model.eval()
all_preds = []
all_labels = []


for batch in test_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)
    with torch.no_grad():#we used this i guess to ignore the gradients or dont work on it tell me exactly for this 
        outputs = model(input_ids = input_ids, attention_mask = attention_mask)
    
    logits = outputs.logits
    preds = torch.argmax(logits, dim = 1)
    
    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())
    
    

In [ ]:
from sklearn.metrics import accuracy_score 

print(accuracy_score(all_labels, all_preds))

1.0


In [ ]:
text = "i absolutely loved this film, it was breathtaking"

inputs = tokenizer(text, return_tensors = "pt", truncation = True, padding = True)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs) 
    
pred = torch.argmax(outputs.logits, dim = 1).item()
print("positive" if pred == 1 else "Negative")

Negative


In [ ]:
from transformers import EarlyStoppingCallback

In [ ]:
from transformers import TrainingArguments 

training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 3, 
    per_device_train_batch_size = 8, 
    per_device_eval_batch_size = 8, 
    eval_strategy="epoch",
    load_best_model_at_end = True
)

In [ ]:
import numpy as np 
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis = 1)
    return {"accuracy": accuracy_score(labels, preds)} 



In [ ]:
from transformers import Trainer 

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_obj,
    eval_dataset = test_obj, 
    compute_metrics = compute_metrics
)

In [ ]:
trainer.train()